# GOM ZIP KẾT QUẢ — tải toàn bộ tệp nén thực nghiệm về máy

Notebook **chỉ đọc**. Không train, không xoá, không clone mã. Chạy vài phút.

## Nó làm gì

1. Mount Drive
2. Liệt kê mọi `.zip` trong `MyDrive/mobivital/` — trừ `by_user.tar` và
   `windows.tar.gz` (2,7 GB, không cần)
3. Với mỗi zip: chỉ lấy `summary.csv`, `scores_*.csv`, `curve.csv`, `*.txt`,
   `README.txt` — **bỏ `final.pth`** (trọng số model, nặng nhất, không dùng cho
   luận văn). Muốn giữ cả `final.pth` thì đổi `BO_CHECKPOINT = False` ở mục 3.
4. Gộp tất cả thành **một tệp** `ket_qua_thuc_nghiem.tar.gz`
5. Bấm tải về máy

Gửi lại link notebook đã chạy + tải tệp về, tôi sắp vào thư mục.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Liệt kê mọi tệp nén thực nghiệm

In [ ]:
import glob, os, subprocess

SRC = "/content/drive/MyDrive/mobivital"
BO  = {"by_user.tar", "windows.tar.gz"}          # dữ liệu, không phải kết quả

zips = sorted(p for p in glob.glob(SRC + "/*")
              if p.endswith(".zip") and os.path.basename(p) not in BO)

tong = 0
print("  %-64s %8s   %s" % ("tệp", "MB", "sửa lúc"))
print("  " + "-" * 92)
for p in zips:
    mb = os.path.getsize(p) / 1048576
    tong += mb
    gio = subprocess.run(["date", "-r", p, "+%m-%d %H:%M"],
                         capture_output=True, text=True).stdout.strip()
    print("  %-64s %8.1f   %s" % (os.path.basename(p)[:64], mb, gio))
print("  " + "-" * 92)
print("  %d tệp  ·  tổng %.0f MB" % (len(zips), tong))

## 3. Rút gọn từng zip rồi gộp làm một

`BO_CHECKPOINT = True` bỏ `final.pth` — nhẹ hơn nhiều, đủ cho luận văn (điểm, đường cong loss, bảng lựa chọn kênh vẫn còn).

In [ ]:
import shutil, tarfile, tempfile, zipfile

BO_CHECKPOINT = True
GIU = (".csv", ".txt")          # + .pth nếu BO_CHECKPOINT = False

work = tempfile.mkdtemp()
out_root = os.path.join(work, "ket_qua_thuc_nghiem")
os.makedirs(out_root)

lay = giu_pth = 0
for p in zips:
    ten = os.path.splitext(os.path.basename(p))[0]
    dest = os.path.join(out_root, ten)
    with zipfile.ZipFile(p) as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            name = info.filename
            keep = name.endswith(GIU)
            if not BO_CHECKPOINT and name.endswith(".pth"):
                keep = True
            if not keep:
                continue
            target = os.path.join(dest, name)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with zf.open(info) as s, open(target, "wb") as d:
                shutil.copyfileobj(s, d)
            lay += 1
            if name.endswith(".pth"):
                giu_pth += 1

archive = "/content/ket_qua_thuc_nghiem.tar.gz"
with tarfile.open(archive, "w:gz") as tf:
    tf.add(out_root, arcname="ket_qua_thuc_nghiem")

mb = os.path.getsize(archive) / 1048576
print("  lấy %d tệp con từ %d zip  (giữ %d file .pth)" % (lay, len(zips), giu_pth))
print("  -> %s   %.1f MB" % (archive, mb))

## 4. Xem cây thư mục sẽ tải về

In [ ]:
import subprocess
print(subprocess.run(["bash", "-lc",
      "cd %s && find ket_qua_thuc_nghiem -maxdepth 2 | sort | head -80" % work],
      capture_output=True, text=True).stdout)

## 5. Tải về máy

In [ ]:
from google.colab import files
files.download("/content/ket_qua_thuc_nghiem.tar.gz")

## 6. Giải nén ở máy

```bash
tar -xzf ket_qua_thuc_nghiem.tar.gz
```

Ra thư mục `ket_qua_thuc_nghiem/`, mỗi zip một thư mục con. Gửi tôi đường dẫn.